# Quantum Oracle Sketching — Examples

> Quick-start tour of the QOS library. Each example is a 3–5 cell recipe that calls into the three pipeline modules:
>
> - [`oracle_sketching.py`](oracle_sketching.py) — stage 1 (data loading)
> - [`quantum_ml.py`](quantum_ml.py) — stage 2 (query algorithms)
> - [`classical_shadow.py`](classical_shadow.py) — stage 3 (readout)
>
> For the full theory and step-by-step derivation of any of these, see the deep-dive notebooks: [Boolean / state / streaming](qauntum_oracle_sketching.ipynb), [block encoding](qos_block_encoding.ipynb), [LS-SVM](qos_lssvm.ipynb), [PCA](qos_pca.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from oracle_sketching import (
    ideal_oracle,
    matrix_element_sketch,
    sketched_oracle,
    state_sketch,
    stream_sketch,
)
from quantum_ml import (
    apply_qsvt_polynomial,
    chebyshev_inverse_polynomial,
    quantum_linear_solve,
    quantum_lssvm_solve,
    quantum_pca_top_eigenvector,
)
from classical_shadow import (
    interferometric_shadow_estimate,
    interferometric_shadow_snapshots,
    median_of_means,
    shadow_readout_proxy,
)

np.random.seed(0)

## Example 1 — Boolean phase oracle

Sketch a Boolean phase oracle from $M$ classical samples and verify the operator-norm error scales as $N/\sqrt{M}$ in single-realisation form.

In [ ]:
N = 8
f = np.random.randint(0, 2, size=N)
x_samples = np.random.randint(0, N, size=4000)

V = sketched_oracle(f, x_samples)
O = ideal_oracle(f)

print(f"f = {f.tolist()}")
print(f"||V - O||_2 at M = {x_samples.size}, N = {N}: {np.linalg.norm(V - O, 2):.3e}")

## Example 2 — Real-valued state sketch

Prepare a real unit vector via the Hadamard-test variant of the sketch and check fidelity to the target.

In [ ]:
v = np.random.randn(N)
v /= np.linalg.norm(v)
amp_angle = np.pi / 16
amps = state_sketch(v, np.random.randint(0, N, size=5000), amp_angle)

# Recover state in the small-θ limit and check overlap.
v_est = amps / (amp_angle / 2) * np.sqrt(N)
v_est /= np.linalg.norm(v_est)
print(f"|⟨v_est | v⟩| = {abs(v_est @ v):.4f}    (1 = perfect)")

## Example 3 — Block-encoded matrix loading

Sketch the matrix-element phase oracle of a random sparse symmetric matrix and recover its entries.

In [ ]:
A = np.random.uniform(-1, 1, (4, 4))
A = (A + A.T) / 2
A /= np.max(np.abs(A))

theta_be = 0.05
samples_ij = np.column_stack([
    np.random.randint(0, 4, size=80_000),
    np.random.randint(0, 4, size=80_000),
])
V_mat = matrix_element_sketch(A, samples_ij, theta_be)
A_recovered = np.angle(np.diag(V_mat)).reshape(4, 4) / theta_be
A_recovered = 0.5 * (A_recovered + A_recovered.T)

print(f"||A_recovered - A|| / ||A|| = {np.linalg.norm(A_recovered - A) / np.linalg.norm(A):.3e}")

## Example 4 — Linear system $Ax = b$ via QSVT

Solve a small linear system end-to-end through the QOS pipeline and compare to `np.linalg.solve`.

In [ ]:
# Well-conditioned A.
A_lin = np.diag([0.6, 0.7, 0.8, 0.9]) + 0.1 * np.random.randn(4, 4)
A_lin = (A_lin + A_lin.T) / 2
A_lin /= np.linalg.norm(A_lin, 2)
b = np.random.randn(4)

x_quantum, _ = quantum_linear_solve(A_lin, b, M_matrix=80_000, rng=np.random.default_rng(1))
x_classical = np.linalg.solve(A_lin, b)

print(f"||x_quantum - x_classical|| / ||x_classical|| = "
      f"{np.linalg.norm(x_quantum - x_classical) / np.linalg.norm(x_classical):.3e}")

## Example 5 — LS-SVM classification

Train a least-squares SVM on a toy 2-class dataset and compare quantum vs classical accuracy.

In [ ]:
# Toy 2-class blobs.
D, N_TRAIN, N_TEST, LAMBDA = 8, 24, 64, 0.5
mu_dir = np.random.randn(D); mu_dir /= np.linalg.norm(mu_dir)
def make_blobs(n):
    labels = np.random.choice([-1, 1], size=n)
    return mu_dir[None, :] * labels[:, None] + 0.5 * np.random.randn(n, D), labels
X_tr, y_tr = make_blobs(N_TRAIN)
X_te, y_te = make_blobs(N_TEST)
sc = max(np.max(np.abs(X_tr)), np.max(np.abs(X_te)))
X_tr, X_te = X_tr / sc, X_te / sc

# Quantum solver.
w_q, _ = quantum_lssvm_solve(X_tr, y_tr, LAMBDA, rng=np.random.default_rng(2))
# Classical baseline.
w_c = np.linalg.solve(X_tr.T @ X_tr + LAMBDA * np.eye(D), X_tr.T @ y_tr)

acc_q = float(np.mean(np.sign(X_te @ w_q) == y_te))
acc_c = float(np.mean(np.sign(X_te @ w_c) == y_te))
print(f"quantum accuracy:   {acc_q:.3f}")
print(f"classical accuracy: {acc_c:.3f}")
print(f"cosine(w_q, w_c):   {abs(w_q @ w_c) / (np.linalg.norm(w_q) * np.linalg.norm(w_c)):.4f}")

## Example 6 — PCA top eigenvector

Recover the top principal direction of a low-rank-plus-noise feature matrix and compare explained variance.

In [ ]:
# Low-rank + noise feature matrix.
w_true = np.random.randn(D); w_true /= np.linalg.norm(w_true)
amps_true = 1.0 * np.random.randn(N_TEST)  # using N_TEST rows from above for variety
X_pca = amps_true[:, None] * w_true[None, :] + 0.15 * np.random.randn(N_TEST, D)
X_pca /= np.max(np.abs(X_pca))
X_pca = X_pca - X_pca.mean(axis=0, keepdims=True)

# Classical baseline (top eigenvector of X_pca.T @ X_pca).
eigvals, eigvecs = np.linalg.eigh(X_pca.T @ X_pca)
w_classical_pca = eigvecs[:, -1]
lambda_top, lambda_second = float(eigvals[-1]), float(eigvals[-2])

# Guiding vector — noisy initial estimate.
g = w_true + 0.5 * np.random.randn(D); g /= np.linalg.norm(g)

w_quantum_pca, _ = quantum_pca_top_eigenvector(
    X_pca, g,
    spectral_gap_lower=lambda_second / lambda_top,
    spectral_gap_upper=1.0,
    rng=np.random.default_rng(3),
)

print(f"|⟨w_quantum | w_classical⟩| = {abs(w_quantum_pca @ w_classical_pca):.4f}")
ev_classical = float(np.var(X_pca @ w_classical_pca, ddof=1))
ev_quantum = float(np.var(X_pca @ w_quantum_pca, ddof=1))
print(f"explained variance (classical): {ev_classical:.3f}")
print(f"explained variance (quantum):   {ev_quantum:.3f}")

## Example 7 — Interferometric-shadow readout

Predict test-set labels from the quantum LS-SVM weight via the sign-preserving shadow proxy. Just $K = 200$ "shots" suffice to match the classical baseline.

In [ ]:
# Reuse the LS-SVM solution from Example 5.
preds_shadow = shadow_readout_proxy(w_q, X_te, K=200, rng=np.random.default_rng(4))
acc_shadow = float(np.mean(preds_shadow == y_te))
print(f"shadow-proxy accuracy (K=200 shots): {acc_shadow:.3f}")
print(f"classical baseline accuracy:         {acc_c:.3f}")

# Optional: median-of-means estimator on a noisy stream.
estimators = X_te @ w_q + 0.3 * np.random.randn(X_te.shape[0])
mom_est = median_of_means(estimators, k=5)
print(f"median-of-means of ⟨x_test|w⟩ stream: {mom_est:.3f}")

## Example 8 — Streaming view

Watch the sketched Boolean oracle converge to the target as samples arrive one at a time. `stream_sketch` keeps only running counts in $\mathcal{O}(N)$ memory.

In [ ]:
checkpoints = [100, 250, 1000, 5000, 20_000]
trajectory = stream_sketch(f, max(checkpoints), checkpoints, rng=np.random.default_rng(5))

ts, errs = zip(*trajectory)
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.loglog(ts, errs, "o-")
ax.set_xlabel("Samples processed")
ax.set_ylabel(r"$\|V_t - O\|_2$")
ax.set_title("Boolean phase oracle — streaming convergence")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## Where to go next

* **Theory** — every example above expands into a full deep-dive notebook in this directory:
  - [`qauntum_oracle_sketching.ipynb`](qauntum_oracle_sketching.ipynb) — Boolean phase, real-valued state, streaming.
  - [`qos_block_encoding.ipynb`](qos_block_encoding.ipynb) — matrix-element + matrix-index oracle theory + QSVT primer.
  - [`qos_lssvm.ipynb`](qos_lssvm.ipynb) — full LS-SVM derivation, sample-complexity scaling, Classiq fragment.
  - [`qos_pca.ipynb`](qos_pca.ipynb) — full PCA derivation, spectral-projector polynomial, Classiq fragment.

* **Module reference** — the public API of each module:
  - [`oracle_sketching.py`](oracle_sketching.py) — see ``__all__`` for the full list.
  - [`quantum_ml.py`](quantum_ml.py) — QSVT helpers, linear / LS-SVM / PCA solvers.
  - [`classical_shadow.py`](classical_shadow.py) — interferometric + Pauli shadows, MoM, error bounds.

* **What's deferred** — full end-to-end Classiq execution at $D \ge 8$, real-data demos (IMDB, PBMC), dynamic-data extensions (Theorems 2/4/6 of [[1]](#ref1)). See the `## Out of scope` section of each deep-dive notebook for the exact boundaries.

## References

<a id='ref1'></a>
[[1]](#ref1) Zhao, H., Zlokapa, A., Neven, H., Babbush, R., Preskill, J., McClean, J. R., and Huang, H.-Y. *Exponential quantum advantage in processing massive classical data.* arXiv:2604.07639 (2026). https://arxiv.org/abs/2604.07639

<a id='ref2'></a>
[[2]](#ref2) Huang, H.-Y., Kueng, R., and Preskill, J. *Predicting many properties of a quantum system from very few measurements.* Nature Physics 16, 1050–1057 (2020). arXiv: https://arxiv.org/abs/2002.08953

<a id='ref4'></a>
[[4]](#ref4) Gilyén, A., Su, Y., Low, G. H., and Wiebe, N. *Quantum singular value transformation and beyond.* STOC 2019. arXiv: https://arxiv.org/abs/1806.01838